### Setup and Library Imports

The following libraries are imported to support data manipulation and visualization throughout this notebook. `pandas` and `numpy` are used for data handling, while `matplotlib` and `seaborn` provide visualization capabilities. Global plotting settings are configured to ensure consistent and readable chart formatting across all sections.

In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# 1. Introduction & Problem Statement

In today’s world, many individuals and organizations rely on loans to support important financial needs such as education, housing, or business activities. Because of this, financial institutions receive a large number of loan applications from people with different financial backgrounds.
Deciding whether an applicant should be approved for a loan is an important process, as banks must evaluate the applicant’s ability to repay while minimizing financial risk. This decision usually depends on several factors such as income, credit score, employment status, loan amount, and asset values.
However, manually evaluating a large number of applications can be complex and time-consuming. Machine learning techniques can help address this challenge by analyzing historical loan data and identifying patterns that influence loan approval decisions.
In this project, we analyze a dataset containing information about loan applicants and their approval outcomes. We begin with exploratory data analysis to understand the relationships between features and loan approval status. Then, supervised learning models are applied to predict loan approval decisions, while unsupervised learning methods are used to discover patterns in the data. In the final stage, generative AI techniques are integrated to enhance the system by providing explanations and insights based on the model outputs.

Financial institutions must evaluate multiple factors when deciding whether to approve a loan application. Incorrect decisions can lead to financial losses if borrowers fail to repay their loans. Therefore, it is important to analyze applicant information carefully and identify the key attributes that influence loan approval decisions.
We aim to understand the patterns associated with loan approval decisions by exploring the dataset and identifying important relationships between features,that will support the development of machine learning models which is capable of predicting loan approval outcomes.

---

# 2. Dataset Selection & Justification

In our project, we selected the Loan Approval Prediction Dataset available on Kaggle.

 Dataset source: https://www.kaggle.com/datasets/architsharma01/loan-approval-prediction-dataset

 The main purpose of this dataset is to support loan approval prediction, where the goal is to determine whether a loan application should be approved or rejected based on the applicant’s financial and personal attributes. The dataset contains information related to loan applicants such as income, loan amount, credit score (CIBIL), employment status, number of dependents, and asset values. Each record represents a loan application with a corresponding loan approval status, which make it suitable for a classification task.

 This dataset is suitable for our problem for several reasons. First, it is relevant to the domain of financial decision-making since it directly represents the loan approval process used by financial institutions. Second, the dataset provides sufficient information about applicant financial attributes, which are important factors in evaluating loan eligibility. Finally, the dataset is structured and organized, allowing it to be easily analyzed and used for machine learning techniques. These characteristics make the dataset appropriate for exploring patterns related to loan approval decisions and for supporting predictive modeling.

---

# 3. Initial Data Inspection

To understand the structure of the dataset, an initial inspection was performed. This step helps identify the number of observations, feature types, and potential issues such as missing values or inconsistent data types.

The dataset contains **4269 entries** and **13 columns**. Some variables are categorical (such as education and self_employed), while others are numerical (such as income, loan amount, and asset values).

This step ensures that the dataset structure is appropriate before performing further exploratory data analysis.

### Loading the Dataset

The raw dataset, sourced from Kaggle, is loaded into a pandas DataFrame. Upon loading, the CSV file is found to contain extraneous leading whitespace in both column headers and categorical string values. These are formatting artifacts from the source file, not meaningful properties of the data.

Both issues are corrected immediately after loading. This early correction is necessary because all subsequent sections, including initial inspection and exploratory data analysis, rely on exact string matching. Without it, filtering and grouping operations would silently fail due to invisible whitespace characters, producing empty or misleading results throughout the notebook.

In [94]:
df = pd.read_csv('Dataset/raw_kaggle_data.csv')
print("Raw column names from CSV:", df.columns.tolist())

df.columns = df.columns.str.strip()
print("Standardized column names:", df.columns.tolist())

obj_cols = df.select_dtypes(include=['object']).columns
df[obj_cols] = df[obj_cols].apply(lambda x: x.str.strip())
print("Whitespace stripped from categorical columns:", obj_cols.tolist())

Raw column names from CSV: ['loan_id', ' no_of_dependents', ' education', ' self_employed', ' income_annum', ' loan_amount', ' loan_term', ' cibil_score', ' residential_assets_value', ' commercial_assets_value', ' luxury_assets_value', ' bank_asset_value', ' loan_status']
Standardized column names: ['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']
Whitespace stripped from categorical columns: ['education', 'self_employed', 'loan_status']


In [95]:
df.shape
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 4269 entries, 0 to 4268
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   loan_id                   4269 non-null   int64
 1   no_of_dependents          4269 non-null   int64
 2   education                 4269 non-null   str  
 3   self_employed             4269 non-null   str  
 4   income_annum              4269 non-null   int64
 5   loan_amount               4269 non-null   int64
 6   loan_term                 4269 non-null   int64
 7   cibil_score               4269 non-null   int64
 8   residential_assets_value  4269 non-null   int64
 9   commercial_assets_value   4269 non-null   int64
 10  luxury_assets_value       4269 non-null   int64
 11  bank_asset_value          4269 non-null   int64
 12  loan_status               4269 non-null   str  
dtypes: int64(10), str(3)
memory usage: 433.7 KB


### Dataset Overview

The dataset used in this project contains financial and demographic information about loan applicants. The goal of the dataset is to support the development of a machine learning model that predicts whether a loan application should be approved or rejected.

The dataset consists of **4269 observations (rows)** and **13 features (columns)**. Each row represents an individual loan application, while the columns represent different attributes related to the applicant’s financial condition, employment status, and asset ownership.

The features include both **numerical variables** (such as income, loan amount, credit score, and asset values) and **categorical variables** (such as education level and employment status).

The **target variable** in this dataset is `loan_status`, which indicates the final decision made by the bank regarding the loan application.

The target variable contains two classes:
- **Approved** – the loan application was accepted.
- **Rejected** – the loan application was denied.

This makes the problem a **binary classification problem**, where the goal of the machine learning model is to predict whether a loan application will be approved or rejected based on the applicant’s financial information.

### Feature Description

The dataset contains several features that describe the financial and personal characteristics of loan applicants.

| Feature | Description |
|-------|-------------|
| loan_id | Unique identifier for each loan application |
| no_of_dependents | Number of dependents financially supported by the applicant |
| education | Applicant's education level (Graduate or Not Graduate) |
| self_employed | Indicates whether the applicant is self-employed |
| income_annum | Annual income of the applicant |
| loan_amount | Total amount of loan requested |
| loan_term | Duration of the loan repayment period |
| cibil_score | Credit score indicating the applicant’s creditworthiness |
| residential_assets_value | Estimated value of residential assets owned |
| commercial_assets_value | Estimated value of commercial assets owned |
| luxury_assets_value | Value of luxury assets owned by the applicant |
| bank_asset_value | Total value of assets held in bank accounts |
| loan_status | Target variable indicating whether the loan was approved or rejected |

---

# 4. Exploratory Data Analysis (EDA)

### Statistical Summary

Statistical summaries provide important information about the numerical features in the dataset, including the mean, standard deviation, minimum values, and maximum values.

These statistics help us understand the overall distribution of financial attributes such as income, loan amount, and asset values among loan applicants.

From the statistical summary, we observe that financial attributes such as income and asset values vary significantly across applicants, indicating a diverse financial background among borrowers. The credit score (CIBIL score) is particularly important, as it is expected to have a strong influence on loan approval decisions.

In [97]:
df.describe()

### Correlation Analysis

Correlation analysis is used to measure the relationship between numerical features in the dataset. It helps identify which variables may have stronger relationships with each other and which features may influence the target variable.

Financial attributes such as income, loan amount, and asset values may show positive correlations, as applicants with higher income levels often possess more assets. The credit score is expected to have a strong relationship with the loan approval decision, since financial institutions typically rely on credit history to evaluate loan eligibility.

Understanding these correlations helps in selecting relevant features for building the machine learning model in later stages of the project.

In [98]:
corr_matrix = df.corr(numeric_only=True)
corr_matrix

### Correlation Heatmap of Numerical Features

This visualization shows the correlation between numerical variables in the dataset.

In [99]:
numeric_cols = df.select_dtypes(include=np.number).columns.drop('loan_id')
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation Coefficient'})
plt.title('Correlation Heatmap of Numerical Features', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

**Interpretation**

The correlation heatmap illustrates the relationships between numerical features in the dataset. Strong positive correlations are observed between income_annum, loan_amount, and several asset-related variables such as luxury_assets_value and bank_asset_value. This suggests that applicants with higher income levels tend to request larger loans and typically possess higher-value assets.

These strong correlations indicate that many financial variables capture similar aspects of an applicant’s financial capacity. As a result, some features may contain overlapping information, which could introduce redundancy during model training. This should be considered during feature selection or model regularization to avoid potential multicollinearity issues.

In contrast, variables such as cibil_score, loan_term, and no_of_dependents exhibit weak correlations with most financial features. This suggests that these variables provide independent information about applicants and may contribute unique predictive value when building machine learning models for loan approval prediction.

### Class Distribution Analysis

Understanding the distribution of the target variable (`loan_status`) is critical for classification tasks. An imbalanced dataset may require special techniques such as oversampling, undersampling, or class-weighted models to avoid biased predictions.

The following plots show the distribution of approved vs. rejected loans both as counts and percentages.

In [100]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = df['loan_status'].value_counts()
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', linewidth=0.8)
for i, (label, val) in enumerate(zip(class_counts.index, class_counts.values)):
    axes[0].text(i, val + 30, str(val), ha='center', fontweight='bold', fontsize=13)
axes[0].set_title('Loan Status Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Applications')
axes[0].set_xlabel('Loan Status')

axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.03, 0.03),
            textprops={'fontsize': 13, 'fontweight': 'bold'},
            wedgeprops={'edgecolor': 'black', 'linewidth': 0.8})
axes[1].set_title('Loan Status Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Class balance ratio: {class_counts.min() / class_counts.max():.3f}")
print(f"Approved: {class_counts.get('Approved', 0)} | Rejected: {class_counts.get('Rejected', 0)}")

Class balance ratio: 0.607
Approved: 2656 | Rejected: 1613


**Interpretation**

The class distribution analysis visualizes the proportion of approved and rejected loan applications in the dataset. The results show that 62.2% of applications were approved, while 37.8% were rejected.

Although the dataset is not perfectly balanced, the difference between the two classes is moderate. This suggests that the dataset does not suffer from severe class imbalance. However, the imbalance should still be considered during model evaluation to ensure that the classifier does not favor the majority class.

### Categorical Features vs. Loan Status

Examining how categorical features (education and self-employment status) relate to the target variable helps identify whether these factors influence loan approval decisions. This cross-tabulation analysis reveals potential patterns in approval rates across different demographic groups.

In [101]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['education', 'self_employed']):
    ct = pd.crosstab(df[col], df['loan_status'])
    ct.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.8)
    ax.set_title(f'Loan Status by {col.replace("_", " ").title()}', fontsize=14, fontweight='bold')
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Count')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(title='Loan Status')

plt.tight_layout()
plt.show()

**Interpretation**

The categorical feature analysis examines how education level and self-employment status relate to loan approval decisions. The results show that both graduates and non-graduates have similar approval and rejection distributions, indicating that education level may not strongly influence loan approval outcomes.

Similarly, the comparison between self-employed and non-self-employed applicants reveals comparable approval patterns. This suggests that self-employment status alone may not significantly affect loan approval decisions in this dataset, and other financial factors such as income, assets, and credit score may play a more important role.

### Feature Distributions

Histograms and box plots help us understand the spread, skewness, and presence of outliers in numerical features. Features with skewed distributions may need transformation during preprocessing, while outliers could affect model performance.

The histograms below are split by loan status (Approved vs. Rejected) to reveal how each feature's distribution differs between the two classes.

In [102]:
features_to_plot = ['income_annum', 'loan_amount', 'loan_term', 'cibil_score',
                     'residential_assets_value', 'commercial_assets_value',
                     'luxury_assets_value', 'bank_asset_value', 'no_of_dependents']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(features_to_plot):
    for status, color in zip(['Approved', 'Rejected'], ['#2ecc71', '#e74c3c']):
        subset = df[df['loan_status'] == status][col]
        axes[i].hist(subset, bins=30, alpha=0.6, label=status, color=color, edgecolor='black', linewidth=0.5)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].legend(fontsize=9)
    axes[i].set_ylabel('Frequency')

fig.suptitle('Feature Distributions by Loan Status', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretation**

The feature distribution analysis compares numerical feature distributions between approved and rejected loan applications. These histograms help identify potential differences between the two classes and reveal patterns that may influence loan approval decisions.

The distributions suggest that CIBIL score plays a strong role in loan approval, as approved applicants generally have higher credit scores while rejected applicants tend to appear more frequently at lower score ranges. Similarly, applicants with higher income and larger asset values appear more frequently among approved loans, indicating that financial strength may positively influence approval decisions.

In contrast, variables such as loan term and number of dependents show similar distributions across both classes, suggesting they may have weaker predictive power for determining loan approval outcomes.

In [103]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(features_to_plot):
    sns.boxplot(data=df, x='loan_status', y=col, ax=axes[i],
                palette={'Approved': '#2ecc71', 'Rejected': '#e74c3c'})
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')

fig.suptitle('Box Plots of Features by Loan Status', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretation**

The box plots compare the distributions of numerical features between approved and rejected loan applications. These plots highlight differences in medians, spread, and the presence of outliers across the two classes.

Among all variables, CIBIL score shows the clearest separation between approved and rejected applications. Approved loans generally correspond to higher credit scores, while rejected loans tend to appear at lower score ranges. This suggests that credit score is likely a strong predictor of loan approval.

Other features such as income and asset values show overlapping distributions between the two classes, indicating that while they may contribute to the decision process, they are not as strong individually as the credit score. Variables such as loan term and number of dependents exhibit similar distributions across both classes, suggesting weaker influence on loan approval outcomes.

### Missing Value Analysis

Identifying missing values is a crucial preprocessing step. Missing data can introduce bias and reduce model performance if not handled properly. The visualization below shows the count and percentage of missing values for each feature, along with a matrix view of data completeness.

In [104]:
fig, ax = plt.subplots(figsize=(10, 5))

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in missing_df['Missing Count']]
bars = ax.barh(missing_df.index, missing_df['Missing Count'],
               color=colors, edgecolor='black', linewidth=0.5)

ax.set_title('Missing Values per Feature', fontsize=14, fontweight='bold')
ax.set_xlabel('Missing Count')

# FIX: prevent negative axis values
ax.set_xlim(left=0)

for bar, val in zip(bars, missing_df['Missing Count']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

total_missing = missing.sum()
print(f"\nTotal missing values in dataset: {total_missing}")

if total_missing == 0:
    print("The dataset has NO missing values — no imputation is required.")
else:
    print(f"Missing values found in: {missing[missing > 0].index.tolist()}")


Total missing values in dataset: 0
The dataset has NO missing values — no imputation is required.


**Interpretation**

The missing value analysis evaluates the completeness of the dataset before applying preprocessing or modeling steps. The visualizations include a bar chart showing the number of missing values per feature and a matrix visualization illustrating data completeness across all records.

The analysis reveals that the dataset contains no missing values across all features. As a result, no imputation or data cleaning related to missing values is required. This indicates that the dataset is complete and ready for further preprocessing and modeling steps.

### Feature Importance – Preliminary Analysis

A preliminary feature importance analysis helps identify which features have the strongest influence on loan approval decisions before building the full supervised learning model. We use a Random Forest classifier to rank features by their importance scores — features with higher scores contribute more to the model's decision-making process.

Additionally, we examine the point-biserial correlation between each numerical feature and the encoded target variable to quantify linear relationships.

In [105]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

df_temp = df.copy()
le = LabelEncoder()
df_temp['loan_status_encoded'] = le.fit_transform(df_temp['loan_status'])
df_temp['education_encoded'] = le.fit_transform(df_temp['education'])
df_temp['self_employed_encoded'] = le.fit_transform(df_temp['self_employed'])

feature_cols = ['no_of_dependents', 'education_encoded', 'self_employed_encoded',
                'income_annum', 'loan_amount', 'loan_term', 'cibil_score',
                'residential_assets_value', 'commercial_assets_value',
                'luxury_assets_value', 'bank_asset_value']

X_temp = df_temp[feature_cols]
y_temp = df_temp['loan_status_encoded']

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_temp, y_temp)

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

importances.plot(kind='barh', ax=axes[0], color='#3498db', edgecolor='black', linewidth=0.5)
axes[0].set_title('Random Forest Feature Importance', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Importance Score')

from scipy.stats import pointbiserialr
correlations = {}
for col in numeric_cols:
    corr, pval = pointbiserialr(df_temp['loan_status_encoded'], df_temp[col])
    correlations[col] = corr

corr_series = pd.Series(correlations).sort_values()
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_series]
corr_series.plot(kind='barh', ax=axes[1], color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('Point-Biserial Correlation with Loan Status', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Correlation Coefficient')
axes[1].axvline(x=0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.show()

print("\nTop 5 most important features (Random Forest):")
for feat, score in importances.sort_values(ascending=False).head(5).items():
    print(f"  {feat:35s} → {score:.4f}")


Top 5 most important features (Random Forest):
  cibil_score                         → 0.8141
  loan_term                           → 0.0619
  loan_amount                         → 0.0291
  income_annum                        → 0.0179
  luxury_assets_value                 → 0.0171


**Interpretation**

The preliminary feature importance analysis identifies which variables contribute most to predicting loan approval decisions. Using a Random Forest classifier, we estimate the relative importance of each feature in the prediction process, while point-biserial correlation helps quantify the relationship between numerical features and the binary target variable.

The results show that `cibil_score` is by far the most influential feature, significantly outweighing all other variables. This indicates that credit history plays a central role in determining whether a loan application is approved or rejected. Other features such as `loan_term`, `loan_amount`, and `income_annum` contribute modestly, while variables like `education` and `self_employed` appear to have minimal influence.

These findings are consistent with the earlier exploratory analysis, where higher CIBIL scores were strongly associated with approved loans. The sign of the point-biserial correlation depends on how the target variable was encoded, so the magnitude is more important than the direction in this context.

### Key Feature Analysis: CIBIL Score

In credit risk assessment, the CIBIL score is often one of the most influential factors in loan approval decisions.
To better understand its impact, we analyze the distribution of CIBIL scores for approved and rejected loan applications.

By plotting the density distributions of both groups and marking their mean values, we can visually examine how credit
score differences relate to loan approval outcomes.

In [106]:
plt.figure(figsize=(10,6))

sns.kdeplot(
    data=df,
    x='cibil_score',
    hue='loan_status',
    fill=True,
    alpha=0.6,
    linewidth=2
)

plt.title("Impact of CIBIL Score on Loan Status", fontweight='bold')
plt.xlabel("CIBIL Score")
plt.ylabel("Density (Concentration of Applicants)")

mean_approved = df[df['loan_status']=='Approved']['cibil_score'].mean()
mean_rejected = df[df['loan_status']=='Rejected']['cibil_score'].mean()

plt.axvline(mean_approved, linestyle='--', linewidth=2,
            label=f"Mean Approved: {mean_approved:.0f}")

plt.axvline(mean_rejected, linestyle='--', linewidth=2,
            label=f"Mean Rejected: {mean_rejected:.0f}")

plt.legend()
plt.tight_layout()
plt.show()

**Interpretation**

The CIBIL score shows a strong separation between approved and rejected loan applications.
Approved applicants generally have significantly higher credit scores, while rejected applicants
are concentrated in lower score ranges.

The mean CIBIL score for approved loans (~703) is substantially higher than the mean score for
rejected loans (~429), highlighting the importance of credit history in the loan approval process.

This result aligns with the earlier feature importance analysis, where CIBIL score emerged as
the most influential feature in predicting loan approval outcomes.

---

# 5. Preprocessing Pipeline

This section documents all preprocessing steps applied to the dataset, including justifications for each transformation. The goal is to prepare a clean, properly encoded, and normalized dataset ready for supervised learning models.

### Preprocessing Pipeline Overview

| Step | Transformation | Justification |
|------|---------------|---------------|
| 1 | Data cleaning (strip whitespace, drop `loan_id`) | Remove noise from column names and categorical values |
| 2 | Encode categorical and target variables (`education`, `self_employed`, `loan_status`) | ML models require numerical input; binary encoding for classification |
| 3 | Outlier detection and analysis | Identify extreme values that may distort model training |
| 4 | Feature engineering | Derive meaningful ratios to capture financial health (no cibil_category) |
| 5 | Feature scaling (StandardScaler) | Normalize features for distance-based algorithms |
| 6 | Save preprocessed dataset | Export clean data ready for supervised learning |

### Step 1: Data Cleaning

A working copy of the dataset (`df_processed`) is created to preserve the original data used in exploratory analysis. The `loan_id` column is dropped as it is a non-informative unique identifier that does not contribute to prediction. Leading and trailing whitespace is also stripped from all categorical columns to ensure clean, consistent string values for encoding and downstream analysis.

In [107]:
df_processed = df.copy()

df_processed = df_processed.drop('loan_id', axis=1)

print(f"Columns after cleaning: {df_processed.shape[1]}")
print(f"Columns: {list(df_processed.columns)}")

Columns after cleaning: 12
Columns: ['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']


### Step 2: Encode Categorical Variables

Machine learning algorithms require numerical inputs. We apply the following encoding strategy:

- **`education`**: Binary encoding — `Graduate` → 1, `Not Graduate` → 0. This ordinal mapping reflects the natural ordering of education levels.
- **`self_employed`**: Binary encoding — `Yes` → 1, `No` → 0.
- **`loan_status`** (target): Binary encoding — `Approved` → 1, `Rejected` → 0.

In [108]:
df_processed['education'] = df_processed['education'].map({'Graduate': 1, 'Not Graduate': 0})
df_processed['self_employed'] = df_processed['self_employed'].map({'Yes': 1, 'No': 0})
df_processed['loan_status'] = df_processed['loan_status'].map({'Approved': 1, 'Rejected': 0})

print("Encoding applied:")
print(f"  education     — unique values: {df_processed['education'].unique()}")
print(f"  self_employed — unique values: {df_processed['self_employed'].unique()}")
print(f"  loan_status   — unique values: {df_processed['loan_status'].unique()}")
print(f"\nData types after encoding:\n{df_processed.dtypes}")

Encoding applied:
  education     — unique values: [1 0]
  self_employed — unique values: [0 1]
  loan_status   — unique values: [1 0]

Data types after encoding:
no_of_dependents            int64
education                   int64
self_employed               int64
income_annum                int64
loan_amount                 int64
loan_term                   int64
cibil_score                 int64
residential_assets_value    int64
commercial_assets_value     int64
luxury_assets_value         int64
bank_asset_value            int64
loan_status                 int64
dtype: object


### Step 3: Outlier Detection and Analysis

We use the Interquartile Range (IQR) method to identify outliers. This helps us understand the extent of extreme values in each feature. Rather than removing outliers outright (which could discard valid extreme financial cases), we document them for awareness during model training.

In [109]:
numerical_features = df_processed.select_dtypes(include=np.number).columns.drop('loan_status')

outlier_summary = []
for col in numerical_features:
    Q1 = df_processed[col].quantile(0.25)
    Q3 = df_processed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df_processed[col] < lower) | (df_processed[col] > upper)).sum()
    pct = n_outliers / len(df_processed) * 100
    outlier_summary.append({'Feature': col, 'Outliers': n_outliers, '% of Data': round(pct, 2),
                            'Lower Bound': lower, 'Upper Bound': upper})

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outliers', ascending=False)
print("Outlier Summary (IQR Method):\n")
print(outlier_df.to_string(index=False))
print(f"\nDecision: Outliers are retained because extreme financial values represent")
print(f"real-world variability in loan applicant profiles.")

Outlier Summary (IQR Method):

                 Feature  Outliers  % of Data  Lower Bound  Upper Bound
residential_assets_value        52       1.22  -11450000.0   24950000.0
 commercial_assets_value        37       0.87   -8150000.0   17050000.0
        bank_asset_value         5       0.12   -4900000.0   14300000.0
           self_employed         0       0.00         -1.5          2.5
               education         0       0.00         -1.5          2.5
        no_of_dependents         0       0.00         -3.5          8.5
            income_annum         0       0.00   -4500000.0   14700000.0
             cibil_score         0       0.00         10.5       1190.5
               loan_term         0       0.00         -9.0         31.0
             loan_amount         0       0.00  -13000000.0   42200000.0
     luxury_assets_value         0       0.00  -13800000.0   43000000.0

Decision: Outliers are retained because extreme financial values represent
real-world variability in loa

### Step 4: Feature Engineering

We create new engineered features that capture financial ratios and aggregate asset information, which may improve model performance:

- **`loan_to_income_ratio`**: Measures how large the requested loan is relative to the applicant's annual income. Higher ratios indicate greater financial burden.
- **`total_assets`**: Sum of all asset categories, representing the applicant's overall wealth.
- **`asset_to_loan_ratio`**: Compares total assets to the loan amount — a higher ratio indicates stronger collateral backing.
- **`cibil_category`**: Discretization of the continuous `cibil_score` into three categories — **Poor** (0), **Average** (1), and **Good** (2) — using score bins (0–500], (500–700], (700–900], for use in tree-based models and as an ordinal summary of creditworthiness.

In [110]:
df_processed['loan_to_income_ratio'] = df_processed['loan_amount'] / df_processed['income_annum']

df_processed['total_assets'] = (df_processed['residential_assets_value'] +
                                 df_processed['commercial_assets_value'] +
                                 df_processed['luxury_assets_value'] +
                                 df_processed['bank_asset_value'])

df_processed['asset_to_loan_ratio'] = df_processed['total_assets'] / df_processed['loan_amount']

df_processed['cibil_category'] = pd.cut(
    df_processed['cibil_score'],
    bins=[0, 500, 700, 900],
    labels=[0, 1, 2],
    include_lowest=True
)
df_processed['cibil_category'] = df_processed['cibil_category'].astype(int)

print("New engineered features:")
print(df_processed[['loan_to_income_ratio', 'total_assets', 'asset_to_loan_ratio', 'cibil_category']].describe())
print(f"\nDataset shape after feature engineering: {df_processed.shape}")

New engineered features:
       loan_to_income_ratio  total_assets  asset_to_loan_ratio  cibil_category
count           4269.000000  4.269000e+03          4269.000000     4269.000000
mean               2.984807  3.254877e+07             2.231797        1.004451
std                0.595496  1.950656e+07             0.643145        0.814425
min                1.500000  4.000000e+05             0.750000        0.000000
25%                2.464286  1.630000e+07             1.767347        0.000000
50%                3.000000  3.150000e+07             2.142857        1.000000
75%                3.500000  4.720000e+07             2.616216        2.000000
max                4.000000  9.070000e+07             5.666667        2.000000

Dataset shape after feature engineering: (4269, 16)


### Step 5: Feature Scaling (Normalization)

StandardScaler is applied to transform features to have zero mean and unit variance. This is critical for distance-based algorithms (e.g., SVM, KNN) and gradient-based methods (e.g., neural networks) that are sensitive to feature magnitude differences.

We scale only the feature columns, not the target variable.

In [111]:
from sklearn.preprocessing import StandardScaler

feature_columns = df_processed.columns.drop('loan_status')
scaler = StandardScaler()

df_scaled = df_processed.copy()
df_scaled[feature_columns] = scaler.fit_transform(df_processed[feature_columns])

print("Feature scaling applied (StandardScaler).\n")
print("Before scaling (first 3 rows of selected features):")
print(df_processed[['income_annum', 'loan_amount', 'cibil_score', 'total_assets']].head(3))
print("\nAfter scaling (first 3 rows of selected features):")
print(df_scaled[['income_annum', 'loan_amount', 'cibil_score', 'total_assets']].head(3))

Feature scaling applied (StandardScaler).

Before scaling (first 3 rows of selected features):
   income_annum  loan_amount  cibil_score  total_assets
0       9600000     29900000          778      50700000
1       4100000     12200000          417      17000000
2       9100000     29700000          506      57700000

After scaling (first 3 rows of selected features):
   income_annum  loan_amount  cibil_score  total_assets
0      1.617979     1.633052     1.032792      0.930628
1     -0.341750    -0.324414    -1.061051     -0.797198
2      1.439822     1.610933    -0.544840      1.289524


---

# 6. Preprocessed Dataset Ready for Modeling

### Step 6: Save Preprocessed Dataset

The preprocessed dataset (before scaling) is saved to a CSV file for use in the supervised learning phase. Scaling will be applied during model training to avoid data leakage through the train-test split.

In [112]:
df_processed.to_csv('Dataset/preprocessed_loan_data_v2.csv', index=False)
print(f"Preprocessed dataset saved to 'Dataset/preprocessed_loan_data_v2.csv'")
print(f"Shape: {df_processed.shape}")
print(f"\nFinal column list:")
for i, col in enumerate(df_processed.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nTarget variable distribution (encoded):")
print(df_processed['loan_status'].value_counts())
print(f"\nDataset is ready for supervised learning modeling.")

Preprocessed dataset saved to 'Dataset/preprocessed_loan_data_v2.csv'
Shape: (4269, 16)

Final column list:
   1. no_of_dependents
   2. education
   3. self_employed
   4. income_annum
   5. loan_amount
   6. loan_term
   7. cibil_score
   8. residential_assets_value
   9. commercial_assets_value
  10. luxury_assets_value
  11. bank_asset_value
  12. loan_status
  13. loan_to_income_ratio
  14. total_assets
  15. asset_to_loan_ratio
  16. cibil_category

Target variable distribution (encoded):
loan_status
1    2656
0    1613
Name: count, dtype: int64

Dataset is ready for supervised learning modeling.


---

# 7. Key Insights & Challenges

Based on the exploratory data analysis, visualizations, and preprocessing, several important insights were identified.

### Key Findings

1. **The dataset contains a large number of loan applications (4269)**, providing sufficient data for training machine learning models.

2. **The dataset includes a mix of categorical and numerical features** that describe the applicant’s financial situation, employment status, and asset ownership.

3. **Financial attributes such as income, loan amount, and asset values show significant variation** among applicants.

4. **CIBIL Score is the dominant predictor.** Both Random Forest feature importance and point-biserial correlation analysis confirm that `cibil_score` is by far the strongest predictor of loan approval. The box plots show a clear separation between approved and rejected applicants based on credit score.

5. **Moderate class imbalance.** The class distribution shows a split of approximately 62.2% approved and 37.8% rejected, indicating a moderate imbalance. While not severe, this should be considered during model training.

6. **Strong multicollinearity among financial features.** The correlation heatmap reveals high correlations between `income_annum`, `loan_amount`, and `luxury_assets_value` (r > 0.85). This may cause issues for linear models but is less problematic for tree-based methods.

7. **No missing values.** The missing value visualization confirms a complete dataset with zero null entries across all 13 original features, simplifying the preprocessing pipeline.

8. **Categorical features show weak discrimination.** Education level and self-employment status show similar approval rates across their categories, suggesting limited standalone predictive power.

9. **Engineered features enhance the dataset.** Four new features were created (`loan_to_income_ratio`, `total_assets`, `asset_to_loan_ratio`, and `cibil_category`—discretizing credit score into Poor, Average, and Good bands) to capture financial health indicators and creditworthiness summaries that may improve model accuracy.

10. **The dataset is structured and suitable for supervised machine learning** since it includes a clear target variable (`loan_status`) with two classes: Approved and Rejected.


### Challenges

- **High multicollinearity:** The strong correlation between income and asset features could inflate variance in linear models. Dimensionality reduction or feature selection may be needed.

- **Feature scale differences:** Income and asset values range in millions while features like `loan_term` and `no_of_dependents` are single-digit. StandardScaler normalization was applied to address this.

- **Limited feature diversity:** Most features are financial; no behavioral or temporal features are available, which may limit the model’s ability to capture nuanced approval patterns.
